Can we reproduce the single-machine fittings for JET-ILW and AUG from the *original data* to match the paper? Or will we also have a different $\alpha$ coefficient?

Import libraries and data

In [27]:
import numpy as np
from scipy.optimize import curve_fit
import pandas as pd
import os 

ROOT = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT, "resources", "input")
DATA_PATH = os.path.join(DATA_DIR, "TC-26_data.csv")

df = pd.read_csv(DATA_PATH, na_values=["-"])
df = df.map(lambda x: x.strip() if isinstance(x, str) else x) # strip whitespace for CMOD

PLTH    = df["PLTH"].to_numpy(dtype=float)    # loss power [W]
SPLASMA = df["SPLASMA"].to_numpy(dtype=float) # LCFS surface area [m^2]
BT      = df["BT"].to_numpy(dtype=float)      # vacuum B_t at R0 [T]
NEL     = df["NEL"].to_numpy(dtype=float)     # line-averaged n_e [m^-3]
MEFF    = df["PGASA"].to_numpy(dtype=float)   # effective hydrogenic mass [AMU]
TOK     = df["TOK"].to_numpy(dtype=object)    # tokamak name
DIVCON  = df["DIVCON"].to_numpy(dtype=object) # divertor config name
SEL     = df["SELEC2024"] == 1                # SELEC2024 high-density-branch flag


Units, DIVCON mask

In [28]:
Ploss_MW = PLTH / 1e6 # [MW]
ne_20    = NEL  / 1e20 # [10^20 m^-3]

VT_CONFIGS = {"VT", "CORNER"}#, "V5"} #,"V5E", "V5C", "V5L"}
D = np.where(np.isin(DIVCON, list(VT_CONFIGS)), 1.93, 1.0)

mask = SEL

AUG ONLY

In [29]:
m = mask & (TOK == "AUG")

P    = Ploss_MW[m] #* 0.925
Bt   = np.abs(BT[m])
ne   = ne_20[m]
Meff = MEFF[m]
S    = SPLASMA[m]
Dfac = D[m]              # all 1.0 for AUG'
Dfac = np.ones_like(Bt) # force Dfac=1 for all pulses,

def model(X, alpha, beta, eps, zeta):
    Bt, ne, Meff, S, Dfac = X
    return alpha * Bt**beta * ne**eps * (2.0/Meff)**zeta * Dfac * S

X = (Bt, ne, Meff, S, Dfac) # Independent variables

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

alpha, beta, eps, zeta = popt
perr = np.sqrt(np.diag(pcov))

print(f"alpha = {alpha:.4f} pm {perr[0]:.4f}   (paper  0.0314 pm  0.0042)")
print(f"beta  = {beta:.3f} pm {perr[1]:.3f}   (paper 0.689 pm 0.127)")
print(f"eps   = {eps:.3f} pm {perr[2]:.3f}   (paper  0.949  pm 0.0904)")
print(f"zeta  = {zeta:.3f} pm {perr[3]:.3f}   (paper  0.890  pm 0.064)")

P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fit_norm: {rmse_fit:.3f} (paper  0.142)")

alpha = 0.0339 pm 0.0046   (paper  0.0314 pm  0.0042)
beta  = 0.689 pm 0.127   (paper 0.689 pm 0.127)
eps   = 0.949 pm 0.090   (paper  0.949  pm 0.0904)
zeta  = 0.890 pm 0.064   (paper  0.890  pm 0.064)
RMSE_fit_norm: 0.142 (paper  0.142)


JET-ILW ONLY

In [30]:
m = mask & (TOK == "JET")

P    = Ploss_MW[m]# * 1.08135
Bt   = np.abs(BT[m])
ne   = ne_20[m]
Meff = MEFF[m]
S    = SPLASMA[m]
is_VT = np.isin(DIVCON[m], list(VT_CONFIGS))   

def model(X, alpha, beta, eps, zeta, Dfac):
    Bt, ne, Meff, S, is_VT = X
    D = np.where(is_VT, Dfac, 1.0)
    return alpha * Bt**beta * ne**eps * (2.0/Meff)**zeta * D * S

X = (Bt, ne, Meff, S, is_VT) # Independent variables

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

alpha, beta, eps, zeta, Dfac = popt
perr = np.sqrt(np.diag(pcov))

print(f"alpha = {alpha:.4f} pm {perr[0]:.4f}   (paper  0.0687 pm  0.0053)")
print(f"beta  = {beta:.3f} pm {perr[1]:.3f}   (paper 0.474 pm 0.051)")
print(f"eps   = {eps:.3f} pm {perr[2]:.3f}   (paper  1.22  pm 0.043)")
print(f"zeta  = {zeta:.3f} pm {perr[3]:.3f}   (paper  1.11  pm 0.04)")
print(f"Dfac  = {Dfac:.3f} pm {perr[4]:.3f}   (paper  1.73  pm 0.04)")
P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fit_norm: {rmse_fit:.3f} (paper  0.207)")

alpha = 0.0635 pm 0.0050   (paper  0.0687 pm  0.0053)
beta  = 0.474 pm 0.051   (paper 0.474 pm 0.051)
eps   = 1.221 pm 0.043   (paper  1.22  pm 0.043)
zeta  = 1.113 pm 0.038   (paper  1.11  pm 0.04)
Dfac  = 1.728 pm 0.036   (paper  1.73  pm 0.04)
RMSE_fit_norm: 0.207 (paper  0.207)


## Multi-machine TC-26 (Bt) scaling

In [31]:
m = mask

P    = Ploss_MW[m]# * 1.08135
Bt   = np.abs(BT[m])
ne   = ne_20[m]
Meff = MEFF[m]
S    = SPLASMA[m]
is_VT = np.isin(DIVCON[m], list(VT_CONFIGS))   

def model(X, alpha, beta, eps, zeta, Dfac):
    Bt, ne, Meff, S, is_VT = X
    D = np.where(is_VT, Dfac, 1.0)
    return alpha * Bt**beta * ne**eps * (2.0/Meff)**zeta * D * S

X = (Bt, ne, Meff, S, is_VT) # Independent variables

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

alpha, beta, eps, zeta, Dfac = popt
perr = np.sqrt(np.diag(pcov))

print(f"alpha = {alpha:.4f} pm {perr[0]:.4f}   (paper  0.0441 pm  0.0025)")
print(f"beta  = {beta:.3f} pm {perr[1]:.3f}   (paper 0.580 pm 0.039)")
print(f"eps   = {eps:.3f} pm {perr[2]:.3f}   (paper  1.08  pm 0.03)")
print(f"zeta  = {zeta:.3f} pm {perr[3]:.3f}   (paper  0.975  pm 0.032)")
print(f"Dfac  = {Dfac:.3f} pm {perr[4]:.3f}   (paper  1.93  pm 0.04)")
P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fit_norm: {rmse_fit:.3f} (paper  0.238)")

alpha = 0.0451 pm 0.0026   (paper  0.0441 pm  0.0025)
beta  = 0.577 pm 0.039   (paper 0.580 pm 0.039)
eps   = 1.079 pm 0.026   (paper  1.08  pm 0.03)
zeta  = 0.974 pm 0.032   (paper  0.975  pm 0.032)
Dfac  = 1.931 pm 0.038   (paper  1.93  pm 0.04)
RMSE_fit_norm: 0.238 (paper  0.238)
